In [1]:
import numpy as np
import pandas as pd
import sklearn

In [2]:
from sklearn.model_selection import train_test_split

# Criando um dataset de exemplo com Pandas
df = pd.DataFrame({
    'Idade': [22, 45, 30, 52, 40, 25, 60, 35],
    'Salario': [1200, 3500, 2100, 4800, 3100, 1500, 5500, 2600],
    'Comprou': [0, 1, 0, 1, 1, 0, 1, 0] # Target (y)
})

X = df.drop(columns=['Comprou']) # Matriz de Features 2D
y = df['Comprou']                # Vetor Alvo 1D

# test_size=0.25 -> 25% para teste, 75% para treino
# random_state=42 -> Garante que a divisão é reprodutível
# stratify=y -> OBRIGATÓRIO em classificação! Mantém a mesma proporção de 0s e 1s em ambos os lados
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas")


Treino: 6 linhas | Teste: 2 linhas


In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 1. Aprende estatísticas E transforma o Treino:
X_train_scaled = scaler.fit_transform(X_train)

# 2. Apenas transforma o Teste com o que aprendeu no treino:
X_test_scaled = scaler.transform(X_test)

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# A. Tratar Valores em Falta (NaN)
imputer = SimpleImputer(strategy='median') # ou 'mean', 'most_frequent'
# X_imputado = imputer.fit_transform(X_com_nulos)

# B. One-Hot Encoding (Variáveis Categóricas em 0 e 1)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# handle_unknown='ignore': Se aparecer uma categoria nova no teste, preenche com zeros sem dar erro!
cidades = np.array([['Lisboa'], ['Porto'], ['Coimbra'], ['Lisboa']])
cidades_codificadas = encoder.fit_transform(cidades)

print(cidades_codificadas)

[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]


In [7]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import root_mean_squared_error, r2_score

# 1. Instanciar
modelo_reg = Ridge(alpha=1.0) # Ridge adiciona regularização L2 contra sobreajuste

# 2. Treinar
modelo_reg.fit(X_train_scaled, y_train)

# 3. Prever no conjunto de teste
y_pred = modelo_reg.predict(X_test_scaled)

# 4. Avaliar
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Erro Médio (RMSE): {rmse:.2f}")
print(f"Coeficiente R² (0 a 1): {r2:.2f}")

# Inspecionar os pesos aprendidos:
print("Pesos das Features:", modelo_reg.coef_)
print("Interceto (Bias):", modelo_reg.intercept_)


Erro Médio (RMSE): 0.38
Coeficiente R² (0 a 1): 0.43
Pesos das Features: [0.23163113 0.17906407]
Interceto (Bias): 0.49999999999999994


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Vamos usar uma Floresta Aleatória com 100 árvores
clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# Treinar
clf.fit(X_train, y_train) # Não precisa de escala prévia!

# 1. Previsão de Classes (0 ou 1)
y_pred_classe = clf.predict(X_test)

# 2. Previsão de Probabilidades (Crucial para escolher limiares de decisão)
y_pred_prob = clf.predict_proba(X_test)
print("Probabilidade [P(Classe 0), P(Classe 1)] da 1ª amostra:\n", y_pred_prob[0])

Probabilidade [P(Classe 0), P(Classe 1)] da 1ª amostra:
 [0.99 0.01]


In [9]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Relatório Completo: Precision, Recall e F1-Score por classe
print(classification_report(y_test, y_pred_classe))

# Matriz de Confusão
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred_classe))

# ROC-AUC (Avalia a capacidade do modelo em separar as classes de 0.5 a 1.0)
# Usa sempre as probabilidades da classe positiva: y_pred_prob[:, 1]
auc = roc_auc_score(y_test, y_pred_prob[:, 1])
print(f"Área sob a Curva ROC (AUC): {auc:.3f}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

Matriz de Confusão:
 [[1 0]
 [0 1]]
Área sob a Curva ROC (AUC): 1.000
